# Minimal-Cost Password-Leak Attacks

This notebook uses the three command-line entry scripts documented in the README:

- `scenario_preprocessor.py` to generate/verify scenarios and manifests,
- `pareto_comparison.py` to compare breaking budgets,
- `attack_tree_extractor.py` to render capability-annotated attack trees for `no pw leakage`.

In [1]:
# Notebook is located in notebooks/, so script paths are prefixed with ../
!python ../scenario_preprocessor.py --check-all-scenarios ../examples/hashed_passwords_paper.pv ../examples/singularized_passwords_paper.pv


Processing: ../examples/hashed_passwords_paper.pv
Attacker capabilities with costs
------------------------------------------------------------
Capability                                    hack         time
---------------------------------------------------------------
Rainbow table attack                             -            2
Server compromised                               4            -
Database leak                                    1            -

Total scenarios generated: 8

Processing: ../examples/singularized_passwords_paper.pv
Attacker capabilities with costs
------------------------------------------------------------
Capability                                    hack         time
---------------------------------------------------------------
Rainbow table attack                             -            4
Server compromised                               4            -
Singularization server compromised               3            -
Database leak                     

In [2]:
# Show where manifests and generated scenarios are available for downstream stages
!ls -1 _scenarios/hashed_passwords_paper
!ls -1 _scenarios/singularized_passwords_paper

base_scenario.pv
database_leak.pv
manifest.json
primitives.pvl
rainbow_table_attack+database_leak.pv
rainbow_table_attack.pv
rainbow_table_attack+server_compromised+database_leak.pv
rainbow_table_attack+server_compromised.pv
server_compromised+database_leak.pv
server_compromised.pv
base_scenario.pv
database_leak.pv
database_leak+singularization_database_leak.pv
manifest.json
primitives.pvl
rainbow_table_attack+database_leak.pv
rainbow_table_attack+database_leak+singularization_database_leak.pv
rainbow_table_attack.pv
rainbow_table_attack+server_compromised+database_leak.pv
rainbow_table_attack+server_compromised+database_leak+singularization_database_leak.pv
rainbow_table_attack+server_compromised.pv
rainbow_table_attack+server_compromised+singularization_database_leak.pv
rainbow_table_attack+server_compromised+singularization_server_compromised+database_leak.pv
rainbow_table_attack+server_compromised+singularization_server_compromised+database_leak+singularization_database_leak.pv
rai

## Pareto fronts for `no pw leakage`

Render the Pareto comparison directly from generated scenario directories.

In [3]:
!python ../pareto_comparison.py _scenarios/hashed_passwords_paper _scenarios/singularized_passwords_paper --query "no pw leakage" --costs time,hack --out-png _gen/pareto_no_pw_leakage.png

Saved Pareto plot: _gen/pareto_no_pw_leakage.png


![Rainbow Table Attack + Database Leak](_gen/pareto_no_pw_leakage.png)

## Attack Trees for `no pw leakage`

Render one hard-coded scenario per model with manifest-backed capability attribution.

In [5]:
!python ../attack_tree_extractor.py _scenarios/hashed_passwords_paper/rainbow_table_attack+database_leak.pv --manifest _scenarios/hashed_passwords_paper/manifest.json --query "no pw leakage" --graphviz-svg _gen

!python ../attack_tree_extractor.py _scenarios/singularized_passwords_paper/rainbow_table_attack+database_leak+singularization_database_leak.pv --manifest _scenarios/singularized_passwords_paper/manifest.json --query "no pw leakage" --graphviz-svg _gen --graphviz-pdf _gen


Capability Analysis
Analyzing capabilities from: _scenarios/hashed_passwords_paper/manifest.json
  Database leak: 5 new clauses
  Server compromised: 1 new clauses
  Rainbow table attack: 1 new clauses


Capability clauses (by capability):

Database leak:
  Clause 28 (Query 1, 2) [initial]: table(passwd(u_2,hashedpw_2,salt_2)) -> attacker((u_2,hashedpw_2,salt_2))

Rainbow table attack:
  Clause 0 (Query 1, 2) [initial]: attacker(hashed(m,s)) && attacker(s) -> attacker(m)

Server compromised:
  (none)
Selected query no pw leakage: no pw leakage

Attack Tree Extraction Summary

Clauses extracted: 58
  Clause 0 [Query 1]: attacker(hashed(m,s)) && attacker(s) -> attacke... [initial]
  Clause 1 [Query 1]: attacker(true) [initial]
  Clause 2 [Query 1]: attacker(v) -> attacker(pk(v)) [initial]
  Clause 3 [Query 1]: attacker(v) && attacker(v_1) -> attacker(hashed... [initial]
  Clause 4 [Query 1]: attacker(false) [initial]
  Clause 5 [Query 1]: attacker(v) && attacker(v_1) -> attacker(egenc(.

![Attack tree baseline](_gen/rainbow_table_attack+database_leak_derivation.svg)

![Attack tree singularized](_gen/rainbow_table_attack+database_leak+singularization_database_leak_derivation.svg)